
# 04. 모델 선택 — 베이스라인 사다리

**목적**: LightGBM 같은 복잡한 모델이 "실제로 도움이 되는지"는 단순한 기준선(baseline) 대비로만 증명할 수 있다. 이 노트북은 `model-selection` 스킬의 **베이스라인 사다리**를 아래에서 위로 쌓아, 검증 구간(2024년 전체)에서 대회 공식 지표(`src/metric.py`)로 채점하고 비교한다.

- **입력**: `data/processed/train_features_v1.parquet`, `test_features_v1.parquet` (`03_features` 산출물), `data/train/scada_vestas_train.csv`, `scada_unison_train.csv` (물리 베이스라인 전용)
- **출력**: 베이스라인 4종(그중 4는 GBDT 3개 알고리즘 비교) + 모델 구조 실험 결과 비교표 (`reports/04_model_selection.md`로 옮겨 정리), 다음 단계(`05_tuning.ipynb`)로 넘길 상위 후보

| 단계 | 베이스라인 | 무엇을 재는가 |
|---|---|---|
| 1 | 시간대×월 평균 예측 (기상 미사용) | "기상 정보 없이 계절·시간 패턴만 알아도 나오는 점수" — 기상 정보의 가치를 재는 바닥선 |
| 2 | 물리 파워커브 (SCADA 파워커브 + 예보 허브높이 풍속) | "ML 없이 도메인 지식(파워커브)만으로 얼마나 가는가" |
| 3 | 선형회귀 (풍속·풍속³·sin/cos 소수 피처) | "가장 단순한 학습 모델의 성능" |
| 4 | GBDT 3종(LightGBM/XGBoost/CatBoost) 기본값 + early stopping | "본 게임의 출발점" — `model-selection` 스킬이 명시한 후보 알고리즘을 동일 조건에서 비교. 이후 튜닝은 이 중 최고 성능 알고리즘 위에서만 |

**"공간자료 아닌가?" 메모**: GFS/LDAPS는 원래 여러 격자에 걸친 공간자료지만, `03_features`에서 터빈별 최근접 격자 가중평균으로 이미 그룹당 시계열 1개로 압축했다(02_eda 확인: GFS는 전 그룹 격자 1개로 수렴, LDAPS도 최대 격자 2개). 격자 간 다양성이 원래 작아서 CNN 같은 공간모델보다 격자 가중치 방식이 더 적합하다고 판단.

**"딥러닝은 왜 안 보나?" 메모**: 이 문제는 예보(외생변수) → 발전량 회귀 구조라 실제 발전량을 lag로 쓰는 자기회귀가 leakage-guard상 불가능하다(평가 기간엔 실제값 자체가 미공개). 데이터도 2.6만 행 규모로 크지 않다. GEFCom 등 실증에서 GBDT가 꾸준히 우세했던 것과 같은 이유로, 이번 노트북은 GBDT 3종만 비교하고 딥러닝은 오류 분석에서 GBDT가 못 잡는 패턴이 드러날 때 별도 실험으로 검토한다.

**leakage-guard 체크**: train/validation 분리는 `HANDOFF.md`에 이미 확정된 기본안을 그대로 재사용한다 (train = 2022-01-01 01:00 ~ 2024-01-01 00:00, validation = 2024-01-01 01:00 ~ 2025-01-01 00:00). 베이스라인2의 SCADA 파워커브는 **train 구간의 SCADA만** 사용해서 만든다 — validation 구간 SCADA를 곡선 학습에 쓰면 "미래 발전 특성"이 곡선에 스며드는 누수이기 때문이다.


## 0. 셋업

패키지를 불러오고, 저장소 루트(REPO_ROOT)를 찾고, 대회 공식 채점기(`src/metric.py`)를 불러온다. `metric.py`는 절대 수정하지 않고 그대로 가져다 쓴다.

In [22]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

SEED = 42
np.random.seed(SEED)

# 어느 위치에서 실행되든 REPO_ROOT를 찾아낸다
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
TRAIN_DIR = REPO_ROOT / "data" / "train"

pd.set_option("display.max_columns", 100)
print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("TARGET_COLS:", TARGET_COLS)
print("CAPACITY_KWH:", CAPACITY_KWH)

python executable: c:\Users\cho03\Desktop\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: c:\Users\cho03\Desktop\wind_forecast
TARGET_COLS: ['kpx_group_1', 'kpx_group_2', 'kpx_group_3']
CAPACITY_KWH: {'kpx_group_1': 21600, 'kpx_group_2': 21600, 'kpx_group_3': 21000}


## 1. 데이터 로드 및 train/validation 분리

`03_features`가 만든 피처 캐시를 불러오고, `HANDOFF.md`에 확정된 분리 기준으로 train/validation을 나눈다.

- **train**: 2022-01-01 01:00 ~ 2024-01-01 00:00 (2년)
- **validation**: 2024-01-01 01:00 ~ 2025-01-01 00:00 (test와 동일한 "1년 전체" 구조라 리더보드와 상관이 맞음)
- `kpx_group_3`은 라벨이 2023년부터만 있어서, train 구간 안에서도 2022년 분은 group_3에는 그냥 결측으로 남는다. `metric()`이 실제값(actual) 기준으로 `actual >= 설비용량*10%`를 채점 대상으로 거르기 때문에, 결측(NaN) 시간대는 비교 시 자동으로 제외된다 — 따로 행을 지우지 않아도 된다.

In [23]:
train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v1.parquet")
test_features = pd.read_parquet(PROCESSED_DIR / "test_features_v1.parquet")

print("train_features:", train_features.shape)
print("test_features:", test_features.shape)
print("기간:", train_features["forecast_kst_dtm"].min(), "~", train_features["forecast_kst_dtm"].max())

train_features: (26304, 55)
test_features: (8760, 53)
기간: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


In [24]:
TRAIN_START = pd.Timestamp("2022-01-01 01:00:00")
TRAIN_END = pd.Timestamp("2024-01-01 00:00:00")
VALID_START = pd.Timestamp("2024-01-01 01:00:00")
VALID_END = pd.Timestamp("2025-01-01 00:00:00")

dtm = train_features["forecast_kst_dtm"]
train_df = train_features[(dtm >= TRAIN_START) & (dtm <= TRAIN_END)].reset_index(drop=True).copy()
valid_df = train_features[(dtm >= VALID_START) & (dtm <= VALID_END)].reset_index(drop=True).copy()

print("train_df:", train_df.shape, train_df["forecast_kst_dtm"].min(), "~", train_df["forecast_kst_dtm"].max())
print("valid_df:", valid_df.shape, valid_df["forecast_kst_dtm"].min(), "~", valid_df["forecast_kst_dtm"].max())
print("겹치는 시각 수(0이어야 함):", len(set(train_df["forecast_kst_dtm"]) & set(valid_df["forecast_kst_dtm"])))
print()
print("그룹별 라벨 결측 (train_df):")
print(train_df[TARGET_COLS].isna().sum())
print("그룹별 라벨 결측 (valid_df):")
print(valid_df[TARGET_COLS].isna().sum())

train_df: (17520, 55) 2022-01-01 01:00:00 ~ 2024-01-01 00:00:00
valid_df: (8784, 55) 2024-01-01 01:00:00 ~ 2025-01-01 00:00:00
겹치는 시각 수(0이어야 함): 0

그룹별 라벨 결측 (train_df):
kpx_group_1      98
kpx_group_2      97
kpx_group_3    8760
dtype: int64
그룹별 라벨 결측 (valid_df):
kpx_group_1    6
kpx_group_2    6
kpx_group_3    6
dtype: int64


## 2. 채점 도구

대회 공식 채점기 `metric(answer_df, pred_df)`는 `forecast_kst_dtm` 순서가 같은 두 DataFrame(`kpx_group_1/2/3` 컬럼 포함)을 받아 `(Score, 1-NMAE, FICR)`를 돌려준다. 매번 이 형태를 손으로 만들지 않도록 헬퍼 함수를 만든다.

- `make_answer_df`: valid_df에서 실제 라벨만 뽑는다
- `make_pred_df`: 예측 딕셔너리(그룹명 → 배열)를 같은 형태의 DataFrame으로 만든다
- `evaluate`: 위 둘을 만들어 `metric()`에 넣고, 결과를 `results` 리스트에 쌓으면서 출력한다

In [25]:
def make_answer_df(df):
    return df[["forecast_kst_dtm", *TARGET_COLS]].reset_index(drop=True)


def make_pred_df(df, pred_dict):
    out = df[["forecast_kst_dtm"]].reset_index(drop=True).copy()
    for col in TARGET_COLS:
        out[col] = np.clip(pred_dict[col], 0, CAPACITY_KWH[col])
    return out


results = []


def evaluate(name, df, pred_dict):
    answer_df = make_answer_df(df)
    pred_df = make_pred_df(df, pred_dict)
    score, one_minus_nmae, ficr = metric(answer_df, pred_df)
    results.append({"baseline": name, "Score": score, "1-NMAE": one_minus_nmae, "FICR": ficr})
    print(f"[{name}] Score={score:.4f}  1-NMAE={one_minus_nmae:.4f}  FICR={ficr:.4f}")
    return score, one_minus_nmae, ficr

## 3. 베이스라인 1 — 시간대×월 평균 예측

**결론 먼저**: 기상 정보를 전혀 쓰지 않고, "1월 3시에는 평균적으로 얼마나 발전했나"라는 표만 만들어 검증 구간에 그대로 복사해 붙인다.

**왜 이걸 먼저 하는가**: 태백 지역은 겨울철 계절풍이 강해서(`02_eda`에서 확인) 월별·시간대별 발전량 패턴 자체가 이미 어느 정도 예측력을 가진다. 이후 나올 모델들이 이 "계절+시간만 아는 모델"보다 못하다면, 기상 예보 피처가 오히려 방해가 되고 있다는 신호다.

In [26]:
train_df["hour"] = train_df["forecast_kst_dtm"].dt.hour
train_df["month"] = train_df["forecast_kst_dtm"].dt.month
valid_df["hour"] = valid_df["forecast_kst_dtm"].dt.hour
valid_df["month"] = valid_df["forecast_kst_dtm"].dt.month

baseline1_pred = {}
for col in TARGET_COLS:
    lookup = train_df.groupby(["month", "hour"])[col].mean()
    keys = list(zip(valid_df["month"], valid_df["hour"]))
    baseline1_pred[col] = np.array([lookup.get(k, lookup.mean()) for k in keys])

print("baseline1_pred 샘플 (kpx_group_1):", baseline1_pred["kpx_group_1"][:5])
print("결측 여부:", any(np.isnan(v).any() for v in baseline1_pred.values()))

baseline1_pred 샘플 (kpx_group_1): [11305.05733871 11996.25632258 12233.26559677 12732.24446774
 12757.90493548]
결측 여부: False


In [27]:
evaluate("1. 시간대x월 평균", valid_df, baseline1_pred)

[1. 시간대x월 평균] Score=0.4336  1-NMAE=0.7483  FICR=0.1189


(np.float64(0.43359610457798464),
 np.float64(0.7483278676379501),
 np.float64(0.11886434151801921))

## 4. 베이스라인 2 — 물리 파워커브

**파워커브(power curve)**란 "풍속이 몇 m/s일 때 터빈이 몇 kWh를 내는가"를 나타내는 곡선이다. 자동차의 속도-연비 그래프처럼, 입력(풍속)과 출력(발전량)의 관계를 그림 하나로 압축한 것 — 이게 풍력발전에서 ML 없이도 쓸 수 있는 가장 강력한 도메인 지식이다.

**만드는 방법**:
1. SCADA 원본(10분 단위, 터빈별 풍속·발전량)을 **train 구간(2022-01-01 01:00~2024-01-01 00:00)만** 잘라서 쓴다 (validation 구간을 곡선 학습에 쓰면 안 됨 — leakage-guard)
2. 터빈별 10분 값을 라벨과 같은 방식으로 1시간 단위로 합친다: 풍속은 평균, 발전량은 합(=1시간 kWh)
3. 그룹 내 터빈들을 합쳐 "그룹 시간당 발전량(kWh, 라벨과 동일 정의)"과 "그룹 평균 풍속"을 만든다
4. 그룹 평균 풍속을 0.5m/s 구간으로 나누고, 구간별 **중앙값** 발전량을 곡선으로 삼는다 (중앙값을 쓰는 이유: 평균은 센서 오류 스파이크에 민감하지만 중앙값은 안 흔들림)

**예보의 "허브높이(117m) 풍속"은 어떻게 근사하는가**: `info.xlsx` 확인 결과 그룹 3개 모두 허브높이가 117m로 동일하다. GFS는 10/80/100m까지만 주므로, 100m 값을 이미 만들어둔 윈드시어 지수(`gfs_shear_alpha`, `03_features`에서 계산)로 117m까지 짧게 외삽한다: `ws_hub = ws100m × (117/100)^α`. 100m→117m은 17m 차이뿐이라 외삽 오차가 작다 (10m에서 외삽하면 107m를 더 가야 해서 오차가 커짐).

**정확히는**: 이 곡선은 SCADA 센서 위치·터빈 개별 특성(웨이크 손실 등)이 뭉뚱그려진 "그룹 평균 거동"이라 개별 터빈 파워커브보다 거칠다. 그래도 ML 없이 어디까지 가는지 재는 바닥선으로는 충분하다.

In [28]:
GROUP_TURBINES = {
    "kpx_group_1": {"maker": "vestas", "ids": [1, 2, 3, 4, 5, 6], "capacity_kw": 3600},
    "kpx_group_2": {"maker": "vestas", "ids": [7, 8, 9, 10, 11, 12], "capacity_kw": 3600},
    "kpx_group_3": {"maker": "unison", "ids": [1, 2, 3, 4, 5], "capacity_kw": 4200},
}

scada_vestas = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig")
scada_vestas["kst_dtm"] = pd.to_datetime(scada_vestas["kst_dtm"])
scada_unison = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig")
scada_unison["kst_dtm"] = pd.to_datetime(scada_unison["kst_dtm"])

print("scada_vestas:", scada_vestas.shape)
print("scada_unison:", scada_unison.shape)

scada_vestas: (157819, 37)
scada_unison: (105264, 16)


In [29]:
def build_group_hourly(df, maker, ids, capacity_kw):
    '''터빈별 10분 SCADA -> 물리범위 클리핑 -> 그룹 시간당 풍속(평균)/발전량(합) 산출.'''
    cap_kw10m = capacity_kw / 6  # 10분 동안 낼 수 있는 최대 에너지(kWh)
    ws_cols, power_cols = [], []
    clipped = df[["kst_dtm"]].copy()
    for i in ids:
        ws_col = f"{maker}_wtg{i:02d}_ws"
        power_col = f"{maker}_wtg{i:02d}_power_kw10m"
        clipped[ws_col] = df[ws_col]
        clipped[power_col] = df[power_col].clip(lower=0, upper=cap_kw10m)
        ws_cols.append(ws_col)
        power_cols.append(power_col)

    hourly_ws = (
        clipped.set_index("kst_dtm")[ws_cols]
        .resample("h", label="right", closed="right").mean()
        .mean(axis=1)
    )
    hourly_power = (
        clipped.set_index("kst_dtm")[power_cols]
        .resample("h", label="right", closed="right").sum()
        .sum(axis=1)
    )
    out = pd.DataFrame({"group_ws": hourly_ws, "group_kwh": hourly_power}).dropna()
    return out.reset_index().rename(columns={"kst_dtm": "forecast_kst_dtm"})


group_hourly = {
    "kpx_group_1": build_group_hourly(scada_vestas, **GROUP_TURBINES["kpx_group_1"]),
    "kpx_group_2": build_group_hourly(scada_vestas, **GROUP_TURBINES["kpx_group_2"]),
    "kpx_group_3": build_group_hourly(scada_unison, **GROUP_TURBINES["kpx_group_3"]),
}

for g, df in group_hourly.items():
    print(g, df.shape, df["forecast_kst_dtm"].min(), "~", df["forecast_kst_dtm"].max())

kpx_group_1 (26304, 3) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
kpx_group_2 (26304, 3) 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
kpx_group_3 (17536, 3) 2023-01-01 01:00:00 ~ 2025-01-01 00:00:00


In [30]:
WS_BIN_WIDTH = 0.5


def fit_power_curve(hourly_df, train_start, train_end):
    fit_df = hourly_df[
        (hourly_df["forecast_kst_dtm"] >= train_start) & (hourly_df["forecast_kst_dtm"] <= train_end)
    ]
    ws_bin = (fit_df["group_ws"] / WS_BIN_WIDTH).round() * WS_BIN_WIDTH
    curve = fit_df.groupby(ws_bin)["group_kwh"].median().sort_index()
    return curve


power_curves = {g: fit_power_curve(df, TRAIN_START, TRAIN_END) for g, df in group_hourly.items()}

for g, curve in power_curves.items():
    print(f"--- {g}: 구간 {len(curve)}개, ws {curve.index.min()}~{curve.index.max()} m/s ---")
    print(curve.head(3))
    print(curve.tail(3))

--- kpx_group_1: 구간 44개, ws 0.0~22.5 m/s ---
group_ws
0.0    0.0
0.5    0.0
1.0    0.0
Name: group_kwh, dtype: float64
group_ws
21.0    17171.0
21.5     7101.0
22.5    10331.0
Name: group_kwh, dtype: float64
--- kpx_group_2: 구간 49개, ws 0.0~25.0 m/s ---
group_ws
0.0    0.0
0.5    0.0
1.0    0.0
Name: group_kwh, dtype: float64
group_ws
23.0    13233.0
23.5     7187.0
25.0     7364.0
Name: group_kwh, dtype: float64
--- kpx_group_3: 구간 45개, ws 0.5~24.0 m/s ---
group_ws
0.5    0.0
1.0    0.0
1.5    0.0
Name: group_kwh, dtype: float64
group_ws
21.5    0.0
22.5    0.0
24.0    0.0
Name: group_kwh, dtype: float64


In [31]:
def apply_power_curve(ws_hub, curve):
    bins = curve.index.to_numpy()
    values = curve.to_numpy()
    return np.interp(ws_hub, bins, values, left=values[0], right=values[-1])


def add_hub_wind_speed(df):
    # 100m -> 117m(허브높이) 윈드시어 외삽. gfs_shear_alpha, gfs_ws100m은 03_features 산출물.
    return df["gfs_ws100m"] * (117 / 100) ** df["gfs_shear_alpha"]


train_df["ws_hub_gfs"] = add_hub_wind_speed(train_df)
valid_df["ws_hub_gfs"] = add_hub_wind_speed(valid_df)

print(train_df["ws_hub_gfs"].describe())

count    17520.000000
mean         4.136835
std          3.328014
min          0.001278
25%          1.966595
50%          3.108473
75%          5.139422
max         27.491176
Name: ws_hub_gfs, dtype: float64


In [32]:
baseline2_pred = {
    g: apply_power_curve(valid_df["ws_hub_gfs"].to_numpy(), power_curves[g]) for g in TARGET_COLS
}
evaluate("2. 물리 파워커브", valid_df, baseline2_pred)

[2. 물리 파워커브] Score=0.3822  1-NMAE=0.6674  FICR=0.0969


(np.float64(0.38215316231232027),
 np.float64(0.6674139519072013),
 np.float64(0.09689237271743922))

## 5. 베이스라인 3 — 선형회귀 (소수 피처)

풍속의 세제곱(발전량 ∝ 풍속³, 파워커브의 물리적 근거)과 풍향 sin/cos, 시간·월 sin/cos만 넣은 아주 단순한 선형회귀다. 그룹마다 바람 노출이 다르므로(터빈 배치·지형) **그룹별로 따로** 학습한다. 입력 피처는 각 그룹 전용 LDAPS 10m 풍속(`03_features`에서 이미 그룹 가중 계산됨) 기준이다 — `02_eda`에서 LDAPS가 GFS보다 발전량과 상관이 높다고 확인했기 때문.

In [33]:
LINEAR_BASE_FEATURES = ["hour_sin", "hour_cos", "month_sin", "month_cos"]

baseline3_pred = {}
baseline3_models = {}
for g in TARGET_COLS:
    group_features = LINEAR_BASE_FEATURES + [
        f"{g}_ldaps_ws10m", f"{g}_ldaps_ws10m_cube", f"{g}_ldaps_wd10m_sin", f"{g}_ldaps_wd10m_cos",
    ]
    fit_rows = train_df[g].notna()
    X_train = train_df.loc[fit_rows, group_features]
    y_train = train_df.loc[fit_rows, g]

    model = LinearRegression()
    model.fit(X_train, y_train)
    baseline3_models[g] = model
    baseline3_pred[g] = model.predict(valid_df[group_features])

evaluate("3. 선형회귀", valid_df, baseline3_pred)

[3. 선형회귀] Score=0.5477  1-NMAE=0.8441  FICR=0.2513


(np.float64(0.5476801082815461),
 np.float64(0.8440752778167829),
 np.float64(0.25128493874630925))

## 6. 베이스라인 4 — GBDT 3종(LightGBM/XGBoost/CatBoost) 기본값 + early stopping

`model-selection` 스킬 2절이 명시한 후보(LightGBM/XGBoost/CatBoost)를 **동일 피처·동일 fold·동일 early-stopping 규칙**으로 비교한다. 이 단계는 **기본 하이퍼파라미터 + early stopping만** 쓴다(튜닝은 05_tuning에서, 최고 성능을 낸 알고리즘 위주로). 손실 함수는 NMAE(절대오차 기반)와 정합적인 **L1(MAE)** 계열을 쓴다. early stopping용 검증셋은 train 구간 안에서 **시간순 마지막 15%**를 떼어 쓴다 — validation(2024년)은 최종 채점에만 쓰고 학습 과정에는 절대 노출하지 않는다.

그룹마다 라벨 스케일과 유효 기간이 달라서(group_3은 2023년만) 세 알고리즘 모두 우선 **그룹별 3모델**로 학습한다. "통합 1모델" 구조는 다음 섹션의 구조 실험에서 (최고 성능 알고리즘 기준으로) 별도로 비교한다.

In [34]:
DROP_COLS = {"forecast_kst_dtm", "forecast_id", "hour", "month", *TARGET_COLS}
FEATURE_COLS = [c for c in train_df.columns if c not in DROP_COLS]
print("피처 개수:", len(FEATURE_COLS))
print(FEATURE_COLS)

피처 개수: 52
['gfs_ws10m', 'gfs_ws80m', 'gfs_ws100m', 'gfs_wd100m_sin', 'gfs_wd100m_cos', 'gfs_ws100m_sq', 'gfs_ws100m_cube', 'gfs_ws850hpa', 'kpx_group_1_ldaps_ws10m', 'kpx_group_1_ldaps_ws10m_sq', 'kpx_group_1_ldaps_ws10m_cube', 'kpx_group_1_ldaps_wd10m_sin', 'kpx_group_1_ldaps_wd10m_cos', 'kpx_group_2_ldaps_ws10m', 'kpx_group_2_ldaps_ws10m_sq', 'kpx_group_2_ldaps_ws10m_cube', 'kpx_group_2_ldaps_wd10m_sin', 'kpx_group_2_ldaps_wd10m_cos', 'kpx_group_3_ldaps_ws10m', 'kpx_group_3_ldaps_ws10m_sq', 'kpx_group_3_ldaps_ws10m_cube', 'kpx_group_3_ldaps_wd10m_sin', 'kpx_group_3_ldaps_wd10m_cos', 'gfs_shear_alpha', 'gfs_rho', 'gfs_ws100m_rho_corrected', 'kpx_group_1_ldaps_rho', 'kpx_group_1_ldaps_ws10m_rho_corrected', 'kpx_group_2_ldaps_rho', 'kpx_group_2_ldaps_ws10m_rho_corrected', 'kpx_group_3_ldaps_rho', 'kpx_group_3_ldaps_ws10m_rho_corrected', 'gfs_temp_diff_2m_850hpa', 'kpx_group_1_ldaps_blh', 'kpx_group_1_ldaps_gust_range_50m', 'kpx_group_2_ldaps_blh', 'kpx_group_2_ldaps_gust_range_50m', 'kp

In [35]:
# 4a. LightGBM
def train_lightgbm_per_group(df, feature_cols, target_col, seed=SEED, early_stop_frac=0.15):
    rows = df[df[target_col].notna()].sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(rows) * early_stop_frac)
    fit_rows, early_rows = rows.iloc[:-n_early], rows.iloc[-n_early:]

    model = lgb.LGBMRegressor(
        objective="l1",
        n_estimators=2000,
        learning_rate=0.05,
        random_state=seed,
        verbose=-1,
    )
    model.fit(
        fit_rows[feature_cols], fit_rows[target_col],
        eval_set=[(early_rows[feature_cols], early_rows[target_col])],
        eval_metric="l1",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    return model


baseline4_models = {}
baseline4_pred = {}
for g in TARGET_COLS:
    model = train_lightgbm_per_group(train_df, FEATURE_COLS, g)
    baseline4_models[g] = model
    baseline4_pred[g] = model.predict(valid_df[FEATURE_COLS])
    print(f"{g}: best_iteration={model.best_iteration_}")

c:\Users\cho03\Desktop\wind_forecast\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


kpx_group_1: best_iteration=155


c:\Users\cho03\Desktop\wind_forecast\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


kpx_group_2: best_iteration=99


c:\Users\cho03\Desktop\wind_forecast\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


kpx_group_3: best_iteration=168


In [36]:
evaluate("4a. LightGBM 기본값", valid_df, baseline4_pred)

[4a. LightGBM 기본값] Score=0.5847  1-NMAE=0.8595  FICR=0.3099


(np.float64(0.5846748015158916),
 np.float64(0.8594968600966537),
 np.float64(0.3098527429351294))

In [37]:
# 4b. XGBoost (LightGBM과 같은 피처·같은 fold·같은 손실(MAE) 규칙)
def train_xgboost_per_group(df, feature_cols, target_col, seed=SEED, early_stop_frac=0.15):
    rows = df[df[target_col].notna()].sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(rows) * early_stop_frac)
    fit_rows, early_rows = rows.iloc[:-n_early], rows.iloc[-n_early:]

    model = xgb.XGBRegressor(
        objective="reg:absoluteerror",
        n_estimators=2000,
        learning_rate=0.05,
        random_state=seed,
        early_stopping_rounds=100,
    )
    model.fit(
        fit_rows[feature_cols], fit_rows[target_col],
        eval_set=[(early_rows[feature_cols], early_rows[target_col])],
        verbose=False,
    )
    return model


baseline4b_models = {}
baseline4b_pred = {}
for g in TARGET_COLS:
    model = train_xgboost_per_group(train_df, FEATURE_COLS, g)
    baseline4b_models[g] = model
    baseline4b_pred[g] = model.predict(valid_df[FEATURE_COLS])
    print(f"{g}: best_iteration={model.best_iteration}")

kpx_group_1: best_iteration=252
kpx_group_2: best_iteration=97
kpx_group_3: best_iteration=220


In [38]:
evaluate("4b. XGBoost 기본값", valid_df, baseline4b_pred)

[4b. XGBoost 기본값] Score=0.5871  1-NMAE=0.8606  FICR=0.3136


(np.float64(0.5871018318725771),
 np.float64(0.8605836412871825),
 np.float64(0.31362002245797177))

In [39]:
# 4c. CatBoost (LightGBM/XGBoost와 같은 피처·같은 fold·같은 손실(MAE) 규칙)
def train_catboost_per_group(df, feature_cols, target_col, seed=SEED, early_stop_frac=0.15):
    rows = df[df[target_col].notna()].sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(rows) * early_stop_frac)
    fit_rows, early_rows = rows.iloc[:-n_early], rows.iloc[-n_early:]

    model = CatBoostRegressor(
        loss_function="MAE",
        iterations=2000,
        learning_rate=0.05,
        random_seed=seed,
        verbose=False,
    )
    model.fit(
        fit_rows[feature_cols], fit_rows[target_col],
        eval_set=(early_rows[feature_cols], early_rows[target_col]),
        early_stopping_rounds=100,
        verbose=False,
    )
    return model


baseline4c_models = {}
baseline4c_pred = {}
for g in TARGET_COLS:
    model = train_catboost_per_group(train_df, FEATURE_COLS, g)
    baseline4c_models[g] = model
    baseline4c_pred[g] = model.predict(valid_df[FEATURE_COLS])
    print(f"{g}: best_iteration={model.get_best_iteration()}")

kpx_group_1: best_iteration=253
kpx_group_2: best_iteration=356
kpx_group_3: best_iteration=143


In [40]:
evaluate("4c. CatBoost 기본값", valid_df, baseline4c_pred)

[4c. CatBoost 기본값] Score=0.5898  1-NMAE=0.8616  FICR=0.3180


(np.float64(0.5898341169822012),
 np.float64(0.8616307149774757),
 np.float64(0.3180375189869267))

## 7. 베이스라인 사다리 비교표

지금까지 쌓은 결과를 한 표로 본다. **1→2→3→4(a/b/c)로 갈수록 점수가 계단식으로 올라가는지**가 핵심 확인 포인트다. 만약 3(선형회귀)이나 4a/4b/4c(GBDT)가 2(물리 파워커브)보다 낮다면, 피처나 학습 방식에 문제가 있다는 뜻이므로 다음 단계로 넘어가기 전에 원인을 봐야 한다. 4a/4b/4c끼리는 동일 조건(같은 피처·같은 fold·같은 손실)이므로 점수 차이가 곧 알고리즘 자체의 우열이다.

In [41]:
results_df = pd.DataFrame(results)
results_df

,baseline,Score,1-NMAE,FICR
0,1. 시간대x월 평균,0.433596,0.748328,0.118864
1,2. 물리 파워커브,0.382153,0.667414,0.096892
2,3. 선형회귀,0.547680,0.844075,0.251285
3,4a. LightGBM 기본값,0.584675,0.859497,0.309853
4,4b. XGBoost 기본값,0.587102,0.860584,0.313620
5,4c. CatBoost 기본값,0.589834,0.861631,0.318038


## 8. 모델 구조 결정 실험

`model-selection` 스킬 4절 기준, 구조를 두 갈래로 실험한다. 우선 **4a(LightGBM) 위에서** 진행하고, 위 비교표에서 4b/4c가 더 높으면 같은 구조를 그 알고리즘으로 옮겨 다시 확인한다(구조의 우열과 알고리즘의 우열은 별개 질문이라 순서상 알고리즘부터 고르고 그 위에서 구조를 본다). 지금까지는 "그룹별 3모델 + 원시 kWh 타깃"이었다(베이스라인4). 여기서는:

1. **통합 1모델**: 그룹을 구분하는 범주형 피처(`group_id`)를 추가하고, 타깃을 **이용률**(발전량 ÷ 설비용량, 0~1 사이 값 — 그룹마다 설비용량이 달라도 같은 잣대로 비교 가능해짐)로 바꿔 데이터를 3배로 합쳐 학습한다. group_3처럼 라벨이 적은 그룹이 이득을 볼 수 있는지 보는 게 목적
2. **연도 가중**: 그룹별 모델(베이스라인4 구조)에 최근 연도(2023) 샘플에 더 큰 가중치를 줘서 재학습 — 최근 패턴이 검증 연도(2024)와 더 가까울 것이라는 가정 검증

In [42]:
GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]


def to_long(df, feature_cols):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


POOLED_FEATURES = FEATURE_COLS + ["group_id"]
train_long = to_long(train_df, FEATURE_COLS)
train_long["group_id"] = pd.Categorical(train_long["group_id"], categories=GROUP_ID_CATEGORIES)

train_long_sorted = train_long.sort_values("forecast_kst_dtm").reset_index(drop=True)
n_early = int(len(train_long_sorted) * 0.15)
fit_rows, early_rows = train_long_sorted.iloc[:-n_early], train_long_sorted.iloc[-n_early:]

pooled_model = lgb.LGBMRegressor(
    objective="l1", n_estimators=2000, learning_rate=0.05, random_state=SEED, verbose=-1,
)
pooled_model.fit(
    fit_rows[POOLED_FEATURES], fit_rows["utilization"],
    eval_set=[(early_rows[POOLED_FEATURES], early_rows["utilization"])],
    eval_metric="l1",
    callbacks=[lgb.early_stopping(100, verbose=False)],
    categorical_feature=["group_id"],
)
print("best_iteration:", pooled_model.best_iteration_)

c:\Users\cho03\Desktop\wind_forecast\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


best_iteration: 435


In [43]:
pooled_pred = {}
for g in TARGET_COLS:
    valid_g = valid_df.copy()
    valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
    util_pred = pooled_model.predict(valid_g[POOLED_FEATURES])
    pooled_pred[g] = util_pred * CAPACITY_KWH[g]

evaluate("5. 통합모델(이용률 타깃)", valid_df, pooled_pred)

[5. 통합모델(이용률 타깃)] Score=0.5927  1-NMAE=0.8646  FICR=0.3207


(np.float64(0.5926700806883558),
 np.float64(0.8646236467957931),
 np.float64(0.3207165145809186))

In [44]:
YEAR_WEIGHT = {2022: 1.0, 2023: 2.0}


def train_lightgbm_weighted(df, feature_cols, target_col, seed=SEED, early_stop_frac=0.15):
    rows = df[df[target_col].notna()].sort_values("forecast_kst_dtm").reset_index(drop=True)
    weights = rows["forecast_kst_dtm"].dt.year.map(YEAR_WEIGHT).fillna(1.0)
    n_early = int(len(rows) * early_stop_frac)
    fit_rows, early_rows = rows.iloc[:-n_early], rows.iloc[-n_early:]
    fit_w, early_w = weights.iloc[:-n_early], weights.iloc[-n_early:]

    model = lgb.LGBMRegressor(
        objective="l1", n_estimators=2000, learning_rate=0.05, random_state=seed, verbose=-1,
    )
    model.fit(
        fit_rows[feature_cols], fit_rows[target_col], sample_weight=fit_w,
        eval_set=[(early_rows[feature_cols], early_rows[target_col])],
        eval_sample_weight=[early_w],
        eval_metric="l1",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    return model


weighted_pred = {}
for g in TARGET_COLS:
    model = train_lightgbm_weighted(train_df, FEATURE_COLS, g)
    weighted_pred[g] = model.predict(valid_df[FEATURE_COLS])

evaluate("6. 그룹별 LightGBM + 연도가중(2023=2배)", valid_df, weighted_pred)

c:\Users\cho03\Desktop\wind_forecast\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[6. 그룹별 LightGBM + 연도가중(2023=2배)] Score=0.5868  1-NMAE=0.8600  FICR=0.3135


(np.float64(0.5867690581091982),
 np.float64(0.8600416007513292),
 np.float64(0.3134965154670673))

In [45]:
results_df = pd.DataFrame(results)
results_df

,baseline,Score,1-NMAE,FICR
0,1. 시간대x월 평균,0.433596,0.748328,0.118864
1,2. 물리 파워커브,0.382153,0.667414,0.096892
2,3. 선형회귀,0.547680,0.844075,0.251285
3,4a. LightGBM 기본값,0.584675,0.859497,0.309853
4,4b. XGBoost 기본값,0.587102,0.860584,0.313620
5,4c. CatBoost 기본값,0.589834,0.861631,0.318038
6,5. 통합모델(이용률 타깃),0.592670,0.864624,0.320717
7,6. 그룹별 LightGBM + 연도가중(2023=2배),0.586769,0.860042,0.313497


## 9. 오류 분석 미리보기

`model-selection` 스킬 5절: 상위 모델(지금까지 중 최고 점수)의 잔차(예측-실제)를 **풍속 구간별**·**시간대별**로 봐서, 어디서 많이 틀리는지 다음 단계(피처/튜닝) 방향을 잡는다. 여기서는 우선 4a(그룹별 LightGBM)를 기준으로 본다 — 위 비교표에서 4b/4c나 구조 실험 쪽이 더 높게 나오면, 이 셀의 `baseline4_pred`를 그 예측으로만 바꿔서 다시 보면 된다.

In [46]:
residual_rows = []
for g in TARGET_COLS:
    mask = valid_df[g].notna()
    residual = baseline4_pred[g][mask.to_numpy()] - valid_df.loc[mask, g].to_numpy()
    ws = valid_df.loc[mask, "gfs_ws100m"].to_numpy()
    hour = valid_df.loc[mask, "hour"].to_numpy()
    residual_rows.append(pd.DataFrame({"group": g, "residual": residual, "abs_error": np.abs(residual), "ws100m": ws, "hour": hour}))

residual_df = pd.concat(residual_rows, ignore_index=True)
residual_df["ws_bin"] = (residual_df["ws100m"] // 2 * 2).astype(int)

print("풍속 구간별 평균 절대오차(kWh):")
print(residual_df.groupby("ws_bin")["abs_error"].mean())
print()
print("시간대별 평균 절대오차(kWh):")
print(residual_df.groupby("hour")["abs_error"].mean())

풍속 구간별 평균 절대오차(kWh):
ws_bin
0      760.424718
2     2145.117385
4     3167.542664
6     3226.123376
8     3315.441940
10    3189.962647
12    2846.929119
14    3380.871196
16    3355.268564
18    3234.412665
20    4006.064013
Name: abs_error, dtype: float64

시간대별 평균 절대오차(kWh):
hour
0     2435.013383
1     2228.675329
2     2345.961155
3     2320.287385
4     2303.283486
5     2366.351341
6     2282.972911
7     2343.674778
8     2204.972653
9     2024.184249
10    1881.377882
11    1744.946645
12    1619.706810
13    1573.281943
14    1594.549431
15    1597.108017
16    1695.195629
17    1815.486842
18    1941.169033
19    2111.291914
20    2220.858277
21    2318.933708
22    2509.868386
23    2534.248584
Name: abs_error, dtype: float64


## 요약

- 베이스라인 1~4(a/b/c)를 같은 validation(2024년 전체) · 같은 채점기(`src/metric.py`)로 비교했다 (결과: 위 `results_df`) — 4a/4b/4c는 동일 피처·동일 fold이므로 GBDT 알고리즘 자체의 우열을 볼 수 있다
- 모델 구조 실험(통합모델 vs 그룹별, 연도가중)을 LightGBM 위에서 비교했다 (알고리즘 승자가 LightGBM이 아니면 같은 구조를 그 알고리즘으로 옮겨 재확인 필요)
- 오류 분석 미리보기로 풍속 구간·시간대별 약점을 확인했다 — 다음 `wind-domain-features`/`feature-selection` 단계에서 이 약점을 겨냥한 피처를 검토한다
- 공간자료·딥러닝 관련 설계 판단은 맨 위 타이틀 셀의 메모 참고 (요약: 공간 차원은 이미 격자가중으로 압축, 딥러닝은 GBDT가 못 잡는 패턴이 보일 때 별도 검토)

**다음 할 일**:
1. 이 노트북을 실제로 실행해서 각 셀의 수치를 확인 (특히 베이스라인 사다리가 계단식으로 오르는지, GBDT 3종 중 어느 알고리즘이 이기는지, 구조 실험 중 어느 쪽이 이기는지)
2. 결과를 `reports/04_model_selection.md`에 Why/How/Result/So-what으로 정리
3. 상위 1~2개 모델(알고리즘/구조/스케일 조합)만 `feature-selection` → `05_tuning.ipynb`로 넘긴다